In [11]:
#%pip install -U langchain langchain-google-genai langchain-community duckduckgo-search

In [13]:
import os
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun

load_dotenv("backend/.env")

#print(os.getenv("GOOGLE_API_KEY") is not None)


llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

search_tool = DuckDuckGoSearchRun()

llm_with_tools = llm.bind_tools([search_tool])




def run_search_agent(user_question):

    response = llm_with_tools.invoke(user_question)

    
    
    #Gemini generates query: "latest major developments in Java in 2026", notice it is not what we inputed
    if response.tool_calls:


        #now we actually call the tool with the query generated by Gemini
        tool_call = response.tool_calls[0]

        query = tool_call["args"]["query"]
        try:
            search_result = search_tool.invoke(query)

            #simulate network failure by calling a function that raises an exception
            #search_result = failing_search(query)

            

            #call gemini again with the duckduckgo search result to get the final answer

            final_response = llm.invoke(
            f"""Answer the user's question using the following web search result.

            User question:
            {user_question}

            Web search result:
            {search_result}
            """
            )

            return final_response.content[0]["text"]
            

        except Exception as e:
            print(f"Error occurred while searching: {e}")
            return "Search failed. Please rely on your internal knowledge if possible."
    else:
        print("No search required")
        #print(response.content)
        return response.content[0]["text"]

    

In [5]:
answer = run_search_agent(
    "What is latest java features?"
)   

print("\nFINAL ANSWER:")
print(answer)


FINAL ANSWER:
As of May 2026, the latest ready-to-use OpenJDK release is **Java 26**.

Here is a breakdown of the current Java landscape based on the provided information:

*   **Latest Release:** **Java 26** is the current general availability release, featuring ten significant enhancements (including four preview features and one incubator feature).
*   **Latest Long-Term Support (LTS) Version:** **Java 25**, released in September 2025, is the current LTS version. It introduced stable features such as structured concurrency and string templates, as well as preview features like value objects.
*   **Upcoming Version:** **Java 27** is currently available as an early-access/release-candidate build.
*   **Previous LTS:** **Java 21** (released in September 2023) remains a widely used LTS version, particularly for projects relying on stable frameworks like Spring Boot 3.x.

**Recommendation:**
*   For **new projects** requiring stability and broad framework support, **Java 21 LTS** is rec

In [6]:
def failing_search(query):
    raise Exception("Simulated network error")

In [7]:
# final_response = llm.invoke(
#     f"""Answer the user's question using the following web search result.

# User question:
# What are the latest major developments in Java in 2026?

# Web search result:
# {search_result}
# """
# )

# print(final_response.content)

In [8]:
# %pip install python-dotenv

In [9]:
# from dotenv import load_dotenv
# import os

# load_dotenv("backend/.env")

# print(os.getenv("GOOGLE_API_KEY") is not None)

In [10]:
# pip install -U ddgs